# A3.3.6: How transactions maintain data integrity

*IB Computer Science 2027, topic A3 Databases, A3.3 Database programming.* HL only. Command term: **describe**. Builds on A3.3.3.

*Booking a kiln firing takes two statements: deduct the credits, then record the booking. Everything on this page exists because of what happens if the second one never runs.*

**How to use this notebook**
- Click a code cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
- A cell that starts with `%%sql` holds plain SQL. A cell with `<<` after `%%sql` saves its result so the check cell can look at it. A cell that creates or changes things shows an empty box. That is normal.
- A **TODO** cell is yours. Each starts with a placeholder line `SELECT 'write your SQL here' AS todo;`. Delete that line when you have written your own SQL.
- The **check** cells print ✓ when you are right and ✗ with the difference when you are not.
- The database lives in memory. If you restart the kernel, run the cells again from the top.

Notes adapted from the ExamEdge page https://examedge.app/learn/ib/a3-3-6

In [ ]:
# Helper functions for the check cells. Run this once. You do not need to read it.
def _rows(var):
    if var not in globals():
        return None
    return [tuple(r) for r in globals()[var]]

def _norm(rows, places=2):
    return [tuple(round(v, places) if isinstance(v, float) else v for v in r) for r in rows]

def _compare(name, got, want, ordered):
    g, w = _norm(got), _norm(want)
    ok = (g == w) if ordered else (sorted(g, key=repr) == sorted(w, key=repr))
    if ok:
        print("✓", name, "is correct.")
    else:
        print("✗", name, "is not right yet.")
        print("   You returned", len(g), "row(s):", g[:6])
        print("   Expected    ", len(w), "row(s):", w[:6])

def check_rows(name, var, want, ordered=False):
    # Compares the rows saved by a "%%sql var <<" cell with the expected rows.
    got = _rows(var)
    if got is None:
        print("✗", name, ": there is no result yet. Run your SQL cell first.")
        return
    _compare(name, got, want, ordered)

def check_db(name, sql, want, ordered=False):
    # Runs a question against the practice database and compares the rows.
    try:
        got = [tuple(r) for r in db.execute(sql).fetchall()]
    except Exception as e:
        print("✗", name, ": my check could not run:", e)
        return
    _compare(name, got, want, ordered)

def check_choice(choice, correct, why_right, why_wrong):
    c = str(choice).strip().upper()
    if c not in "ABCD" or c == "":
        print("Set choice to A, B, C or D, then run this cell again.")
    elif c == correct:
        print("✓ Correct.", why_right)
    else:
        print("✗ Not quite.", why_wrong)

def check_dict(name, got, want):
    bad = [k for k in want if str(got.get(k, "")).strip().upper() != want[k]]
    if not bad:
        print("✓", name, "all correct.")
    else:
        print("✗", name, ":", len(bad), "of", len(want), "not right yet:")
        for k in bad:
            print("    ", k)

**About this notebook's setup.** Transactions need exact control over `BEGIN`, `COMMIT` and `ROLLBACK`, so this notebook uses a small Python helper `run(...)` instead of `%%sql` cells. You write the SQL text, the helper sends the statements one at a time and prints any table that comes back. The database is a real file in a temporary folder, because we will later close it and reopen it.

In [ ]:
import sqlite3, os, tempfile
DB_FILE = os.path.join(tempfile.mkdtemp(), "studio.db")

def connect():
    # isolation_level=None means Python does not start transactions for us. We write BEGIN and COMMIT ourselves.
    return sqlite3.connect(DB_FILE, isolation_level=None)

db = connect()

HISTORY = []

def reset():
    HISTORY.clear()
    if db.in_transaction:
        db.execute("ROLLBACK")
    db.executescript('''
    DROP TABLE IF EXISTS Kiln_Log; DROP TABLE IF EXISTS Member;
    CREATE TABLE Member (MemberID TEXT PRIMARY KEY, MemberName TEXT, KilnCredits INTEGER CHECK (KilnCredits >= 0));
    CREATE TABLE Kiln_Log (LogID TEXT PRIMARY KEY, MemberID TEXT, FiringDate TEXT, CreditsUsed INTEGER,
                           FOREIGN KEY (MemberID) REFERENCES Member (MemberID));
    INSERT INTO Member VALUES ('M1', 'Ines Varga', 40), ('M2', 'Rafi Osei', 12), ('M3', 'Bo Lindqvist', 25);
    INSERT INTO Kiln_Log VALUES ('L1', 'M3', '2027-02-16', 5), ('L2', 'M1', '2027-02-14', 8);
    ''')

def run(text, conn=None):
    # Runs each statement in turn. Prints a table for a SELECT. Stops at the first error and says what happened.
    c = conn or db
    for stmt in [s.strip() for s in text.split(";") if s.strip()]:
        if stmt.startswith("--"):
            stmt = "\n".join(l for l in stmt.splitlines() if not l.strip().startswith("--")).strip()
            if not stmt:
                continue
        HISTORY.append(stmt.upper())
        try:
            cur = c.execute(stmt)
        except Exception as e:
            print("ERROR:", e, "   <- statement:", stmt.splitlines()[0][:60])
            print("The transaction is still open." if c.in_transaction else "No transaction is open.")
            return False
        if cur.description:
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
            print(" | ".join(cols))
            for r in rows:
                print(" | ".join(str(v) for v in r))
    return True

def show():
    run("SELECT * FROM Member; SELECT * FROM Kiln_Log")

reset()
show()

## 1. The problem

A pottery studio charges members in kiln credits. To book a firing, **two** statements must happen: the credit balance goes down, and a row appears in the firing log. Between the two statements there is a gap where the power can fail, the network can drop, or the second statement can simply be invalid.

- If only the first lands, the studio took credits for a firing it has no record of.
- If only the second lands, it booked a firing nobody paid for.

Neither breaks SQL. Both leave the database describing something that never happened.

A **transaction** staples several statements together so the database treats them as **one job**: either the whole staple goes through or none of it does. It removes the in-between state entirely.

## 2. Transaction control language (TCL)

TCL is the fourth family of SQL commands, next to the DDL and DML of A3.3.1.

| Command | Job |
|---|---|
| `BEGIN TRANSACTION` | starts a transaction; the statements after it are held together |
| `COMMIT` | closes the transaction and makes everything permanent |
| `ROLLBACK` | throws away everything since `BEGIN` and closes it, as if it never happened |

> `ROLLBACK` only reaches back as far as the last `BEGIN TRANSACTION`. Once `COMMIT` has run, that transaction is finished and permanent.

## 3. Worked example: a firing that goes wrong, then goes right

**Step 1.** Ines (M1) has 40 credits. Book a 10-credit firing as a transaction that works.

In [ ]:
reset()
run('''
BEGIN TRANSACTION;
UPDATE Member SET KilnCredits = KilnCredits - 10 WHERE MemberID = 'M1';
INSERT INTO Kiln_Log (LogID, MemberID, FiringDate, CreditsUsed) VALUES ('L3', 'M1', '2027-03-02', 10);
COMMIT;
''')
show()

Credits went from 40 to 30 and log row L3 appeared. Both or neither, and this time both.

**Step 2.** Now a booking that fails halfway. The `INSERT` reuses `LogID` `L1`, which already exists, so the primary key refuses it. Run it and look at what the `UPDATE` did.

In [ ]:
reset()
run('''
BEGIN TRANSACTION;
UPDATE Member SET KilnCredits = KilnCredits - 10 WHERE MemberID = 'M1';
INSERT INTO Kiln_Log (LogID, MemberID, FiringDate, CreditsUsed) VALUES ('L1', 'M1', '2027-03-02', 10);
COMMIT;
''')
print()
show()

The error stopped the helper before `COMMIT`. Look: the credits came off (40 became 30) but the log row never appeared, and the transaction is **still open**. That half-finished state is exactly what atomicity exists to prevent.

**Step 3.** Roll back.

In [ ]:
run("ROLLBACK")
show()

Ines is back on 40 credits. Nothing happened, which is the correct outcome for a booking that could not be recorded.

## 4. The four ACID properties

ACID names the four guarantees that make a transaction trustworthy. Each is easiest to remember through what goes wrong without it.

| | Property | Meaning | Without it |
|---|---|---|---|
| **A** | Atomicity | all of it, or none of it. One failed step cancels the lot | Ines is charged 10 credits for a firing that was never booked |
| **C** | Consistency | every constraint (unique keys, checks) still holds when the transaction ends | Rafi books a 15-credit firing with 12 credits and ends on minus 3 |
| **I** | Isolation | transactions running together cannot see or disturb each other's unfinished work | two members book the last Friday slot at once and both are told it is theirs |
| **D** | Durability | after `COMMIT` the change survives a crash or power cut | the studio loses power an hour after Ines books and her firing is gone |

> **Atomicity vs consistency** are the pair most often mixed up. Atomicity is about **how much** of the transaction happens (all or none). Consistency is about **what the result may look like** (the rules still hold). A transaction can be perfectly atomic and still be refused for breaking a constraint.

Let us watch each one.

**Consistency.** Member `Member` has a rule (`CHECK (KilnCredits >= 0)`). Rafi has 12 credits and tries to spend 15.

In [ ]:
reset()
run('''
BEGIN TRANSACTION;
UPDATE Member SET KilnCredits = KilnCredits - 15 WHERE MemberID = 'M2';
INSERT INTO Kiln_Log (LogID, MemberID, FiringDate, CreditsUsed) VALUES ('L3', 'M2', '2027-03-02', 15);
COMMIT;
''')
run("ROLLBACK")
show()

The rule was enforced: the credit update was refused, and the rollback made sure nothing was left half done.

**Isolation.** We open a second connection (a second user, `other`). Our first user starts a transaction but does not commit yet.

In [ ]:
reset()
other = connect()
run("BEGIN TRANSACTION; UPDATE Member SET KilnCredits = 0 WHERE MemberID = 'M1'")
print("User 1 sees:")
run("SELECT MemberName, KilnCredits FROM Member WHERE MemberID = 'M1'")
print("User 2 sees (user 1 has not committed):")
run("SELECT MemberName, KilnCredits FROM Member WHERE MemberID = 'M1'", conn=other)
run("ROLLBACK")
other.close()

User 2 still sees 40. User 1's unfinished work is invisible to everyone else until it is committed.

**Durability.** We commit a change, close the database completely (like a power cut), and open it again.

In [ ]:
reset()
run("BEGIN TRANSACTION; UPDATE Member SET KilnCredits = 99 WHERE MemberID = 'M3'; COMMIT")
db.close()
print("Database closed. Opening it again...")
db = connect()
run("SELECT MemberName, KilnCredits FROM Member WHERE MemberID = 'M3'")

The committed 99 is still there. The promise made at `COMMIT` survived.

## 5. A transaction inside a program

In a real app you do not type `ROLLBACK` by hand. You wrap the statements and roll back on any error. This is the pattern in Python:

In [ ]:
def book_firing(member, log_id, credits):
    try:
        db.execute("BEGIN TRANSACTION")
        db.execute("UPDATE Member SET KilnCredits = KilnCredits - ? WHERE MemberID = ?", (credits, member))
        db.execute("INSERT INTO Kiln_Log (LogID, MemberID, FiringDate, CreditsUsed) VALUES (?, ?, '2027-03-02', ?)", (log_id, member, credits))
        db.execute("COMMIT")
        return "booked"
    except sqlite3.Error as e:
        db.execute("ROLLBACK")
        return "refused (" + str(e) + ")"

reset()
print(book_firing("M1", "L3", 10))
print(book_firing("M2", "L4", 15))
show()

## 6. Your turn

**TODO 1.** Write a transaction that books a firing for Bo (`M3`, 25 credits): deduct **5** credits and add log row `L3` (member `M3`, date `2027-03-05`, 5 credits used). Include `BEGIN TRANSACTION` and `COMMIT`.

In [ ]:
reset()

In [ ]:
run("""
-- TODO 1: BEGIN TRANSACTION; then UPDATE ...; then INSERT ...; then COMMIT
SELECT 'write your SQL here' AS todo;
""")

In [ ]:
check_db("TODO 1 credits", "SELECT KilnCredits FROM Member WHERE MemberID = 'M3'", [(20,)])
check_db("TODO 1 log row", "SELECT LogID, MemberID, CreditsUsed FROM Kiln_Log WHERE LogID = 'L3'", [("L3", "M3", 5)])
print("Transaction closed?", "✓" if not db.in_transaction else "✗ still open, did you forget COMMIT?")

**TODO 2.** This booking will fail, because Rafi (`M2`) only has 12 credits. Write the transaction as in TODO 1 but for Rafi and 15 credits (log `L5`), and after it runs, write the line that **undoes** it. Rafi's balance must end at 12 and `L5` must not exist.

In [ ]:
reset()

In [ ]:
run("""
-- TODO 2: the transaction for Rafi, 15 credits, log L5. It will fail on the UPDATE. Then ROLLBACK.
SELECT 'write your SQL here' AS todo;
""")

In [ ]:
print("✓ TODO 2 ran a ROLLBACK" if any(h.startswith("ROLLBACK") for h in HISTORY) else "✗ TODO 2: no ROLLBACK was run yet.")
print("✓ TODO 2 tried Rafi's booking" if any(h.startswith("UPDATE") and "'M2'" in h for h in HISTORY) else "✗ TODO 2: Rafi's booking was never attempted.")
check_db("TODO 2 balance unchanged", "SELECT KilnCredits FROM Member WHERE MemberID = 'M2'", [(12,)])
check_db("TODO 2 no log row", "SELECT COUNT(*) FROM Kiln_Log WHERE LogID = 'L5'", [(0,)])
print("Transaction closed?", "✓" if not db.in_transaction else "✗ still open, add ROLLBACK")

**TODO 3 (predict first).** Run a committed transaction that deducts 10 credits from Ines (`M1`), **then** run `ROLLBACK`. Write down what you think her balance will be, run it, and compare.

In [ ]:
reset()

In [ ]:
run("""
-- TODO 3: BEGIN TRANSACTION; UPDATE ... minus 10 for M1; COMMIT;
SELECT 'write your SQL here' AS todo;
""")
run("ROLLBACK")
show()

In [ ]:
check_db("TODO 3: the commit was permanent", "SELECT KilnCredits FROM Member WHERE MemberID = 'M1'", [(30,)])

`ROLLBACK` after `COMMIT` has no open transaction to undo, so the 10 credits stay gone. To reverse it you would need a **new** transaction that deliberately adds them back.

**TODO 4: match the property.** Write `"A"`, `"C"`, `"I"` or `"D"` (Atomicity, Consistency, Isolation, Durability) for the property each scenario is about.

In [ ]:
scenarios = {
    "The power fails after COMMIT and the booking is still there on restart": "?",
    "A booking would leave a member on minus 3 credits, so it is refused": "?",
    "If the INSERT fails, the earlier UPDATE is undone as well": "?",
    "Two members book at the same time and neither sees the other's unfinished changes": "?",
}
check_dict("TODO 4", scenarios, {k: v for k, v in zip(scenarios, ["D", "C", "A", "I"])})

## 7. Check your understanding

A transaction updates two rows and then runs `COMMIT`. A moment later the user runs `ROLLBACK`. What happens? **A** both updates are undone, **B** only the second update is undone, **C** nothing is undone, the committed changes are permanent, **D** the whole database reverts to its last backup.

In [ ]:
choice = "?"   # type A, B, C or D
check_choice(choice, "C", "COMMIT ends the transaction and makes its changes permanent, which is durability. ROLLBACK afterwards has no open transaction to undo.",
             "ROLLBACK undoes changes made since the last BEGIN TRANSACTION, and COMMIT ends that transaction. Once COMMIT has run there is nothing left for ROLLBACK to reverse.")

**Examiner note.** Candidates usually understand the concepts. The marks are lost on imprecise language: the four ACID terms get defined in general words that could describe any of them. Weak: *"Atomicity means the transaction is done properly and the data stays correct."* That would fit consistency or durability just as well. Better: *"Atomicity means the transaction is indivisible: either every one of its operations completes, or none of them does. If any part fails the whole transaction is rolled back, so the database is never left in a half-finished state."* Indivisible, all or none, rolled back on failure. Those three details belong to atomicity alone.

## 8. Revision questions

**Q1. Describe what is meant by a transaction in a database.** [2 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

**Q2. Describe the role of atomicity and durability in ensuring reliable processing of transactions.** [4 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)

**Q3. A studio deducts kiln credits and records a firing as two SQL statements. Describe how TCL commands would be used to protect the data, and explain what would happen if the second statement failed.** [6 marks]

Double-click this box and type your answer here, then press Shift+Enter.

> (your answer)